In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime

# 1. Filtra seu lote
ultimo_ts = spark.table("vg_sales.01_bronze.fact_vg_sales_main").select(F.max("ingested_at")).first()[0]
df_raw = spark.table("vg_sales.01_bronze.fact_vg_sales_main").filter(F.col("ingested_at") == ultimo_ts)

#2 Tipagem de dados

df_main = df_raw.select(
    F.sha2(F.concat_ws("||", "title", "console", "release_date"), 256).alias("id_lote"),
    F.sha2(F.lower(F.trim(F.col("console"))), 256).alias("console_hash"),
    F.sha2(F.lower(F.trim(F.col("publisher"))), 256).alias("publisher_hash"),
    F.sha2(F.lower(F.trim(F.col("genre"))), 256).alias("genre_hash"),
    F.col("img").cast("string"),
    F.col("title").cast("string"),
    F.col("console").cast("string"),
    F.col("genre").cast("string"),
    F.col("publisher").cast("string"),
    F.col("developer").cast("string"),
    F.col("critic_score").cast("double"),
    F.col("total_sales").cast("double"),
    F.col("na_sales").cast("double"),
    F.col("jp_sales").cast("double"),
    F.col("pal_sales").cast("double"),
    F.col("other_sales").cast("double"),
    F.col("release_date").cast("date"),
    F.col("last_update").cast("date"),
     F.col("source_file").cast("string"),
    F.col("ingested_at").cast("timestamp")
)

In [0]:
display(df_main)

In [0]:
#Transformando todos os vazios para null
df_main = df_main.replace("", None)

# Agora sim, pode chamar o Pandera ou fazer o dropDuplicates
df_main = df_main.dropDuplicates(["id_lote"])

In [0]:
# Salva em uma localização temporária (Delta) para validation
df_main.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("vg_sales.01_bronze.stg_vg_sales_cleaned")

In [0]:
%sql
DESCRIBE vg_sales.01_bronze.stg_vg_sales_cleaned